# Sampling Diverse Statements and Graphs

## (A) Sampling Network for Structural Diversity
This notebook is used to sample structurally diverse graphs.
We first generate a pool of candidate graphs $N=1\,000$ and compute, for each candidate, a set of summary metrics: 
1. mean degree, 
2. maximum degree
3. the mean shortest-path length, and 
4. Degree of the 0th node (This node is where we place 'Doctor' agent in the final experiment).\*
We then select $K=8$ graphs using a maximin procedure in the z-scored metric space, ensuring coverage of diverse structural regimes.

\* *We enforce an additional constraint that node $i=0$ exhibits high variability in degree across the selected graphs, so that this node does not occupy a similar structural position.*


In [ ]:
import numpy as np
from scipy.spatial.distance import cdist
from sklearn.preprocessing import StandardScaler

from src.core.metrics.network import (
    connected_components,
    global_clustering_coefficient,
    local_clustering_coefficients,
    mean_shortest_path,
)
from src.Network import Network

import sys
import os
sys.path.append(os.path.abspath('..'))


def maximin_select(X: np.ndarray, K: int) -> list[int]:
    """Select K points from X using the maximin criterion."""
    N = X.shape[0]
        
    selected = [int(np.argmax(np.linalg.norm(X, axis=1)))]
    
    for _ in range(K - 1):
        distances = cdist(X, X[selected], metric="euclidean")
        min_dist = distances.min(axis=1)
        min_dist[selected] = -np.inf  # exclude already selected
        next_idx = np.argmax(min_dist)
        selected.append(int(next_idx))
    
    return selected

# Deterministic seeds
master_rng = np.random.Generator(np.random.PCG64(814183))
seeds = master_rng.integers(
    low=0,
    high=2**32,
    size=1000,
    dtype=np.uint32
)

def graph2vec(G: Network) -> tuple[np.ndarray, int]:
    """Convert a Network object to its adjacency matrix vector representation.

    Args:
        G (Network): The input network.

    Returns:
        np.ndarray: The vector representation of the graph. (mean shortest path, global clustering coefficient, mean degree, std degree, degree of node 0)
    """
    A = G.adjacency_matrix()
    mean_sp = mean_shortest_path(A)
    gcc = global_clustering_coefficient(A)
    std_deg = np.std(np.sum(A, axis=1))
    mean_deg = np.mean(np.sum(A, axis=1))
    nod0_deg = np.sum(A[0, :])

    return np.array([mean_sp, gcc, mean_deg, std_deg, nod0_deg]), len(connected_components(A))


In [ ]:
# GLOBAL PARAMETERS
N = 48

### (A) 1. Sampling `Erdős–Rényi` graphs
We sample ER graph to get diversity based on:
1. Mean shortest path
2. Global Clustering Coefficient
3. Mean Degree 
4. Standard Dev of Degrees
5. Degree of the 0-th node

In [ ]:
cfg_er = {
    "seed": None,
    "network":
        {"generator": "ER",
         "params": {
        "n": N,
        "p": 0.3
         }
        }
}

In [ ]:
X = []
check = []
for seed in seeds:
    cfg = cfg_er.copy()
    cfg["seed"] = int(seed)
    G = Network(cfg, remap_seed=False)
    vec, n_components = graph2vec(G)
    X.append(vec)
    check.append(n_components)
X = np.array(X)
# Scale
Xz = StandardScaler().fit_transform(X)

selection_ids = maximin_select(Xz, K=8)
pass_the_check = all(check[i] == 1 for i in selection_ids)
print(f"All selected graphs are connected: {pass_the_check}")

for i in selection_ids:
    r = X[i]
    print(f'{seeds[i]:>11} : mean-sp={r[0]:.3f}, glob-clustering-coeff={r[1]:.3f}, mean-deg={r[2]:>6.3f}, std-deg={r[3]:.3f}, nod0-deg={r[4]:.3f}')
    
print("\nSelected seeds:")
seeds[selection_ids].tolist()


### (A) 2. Sampling `Watts–Strogatz` Graphs
#### 2.1 Calibrate Watts–Strogatz parameters (k, $\beta$) for small-world sampling

In [ ]:
cfg_ws = {
    "seed": None,
    "network":
        {"generator": "WS",
         "params": {
        "n": N,
        "k": None,
        "beta": None
         }
        }
}
k_grid = [4, 6, 8, 10, 12]
b_grid = [0.001, 0.003, 0.01, 0.03, 0.05, 0.1]
n_samples = 100
thr_connectivity = 0.99
results = []
gammas = []
lambdas = []
for k in k_grid:
    for b in b_grid:
        cfg_ws["network"]["params"]["k"] = k
        cfg_ws["network"]["params"]["beta"] = b
        
        C_ws = []
        L_ws = []
        C_er = []
        L_er = []
        ws_connected = 0
        er_connected = 0
        # Sampling
        for i in range(n_samples):
            cfg_ws["seed"] = int(i)
            cfg_er["seed"] = int(i + 10_000)
            A_ws = Network(cfg_ws, remap_seed=False).adjacency_matrix()
            c_ws = np.mean(local_clustering_coefficients(A_ws))
            l_ws = mean_shortest_path(A_ws)
            
            if l_ws is not None and np.isfinite(l_ws):
                C_ws.append(c_ws)
                L_ws.append(l_ws)
                ws_connected += 1
                
            cfg_er['network']['params']['p'] = k / (N - 1)  # adjust p to match mean degree
            A_er = Network(cfg_er, remap_seed=False).adjacency_matrix()
            c_er = np.mean(local_clustering_coefficients(A_er))
            l_er = mean_shortest_path(A_er)
            
            if l_er is not None and np.isfinite(l_er):
                C_er.append(c_er)
                L_er.append(l_er)
                er_connected += 1
        # Compute connectivity ratios
        ws_connected_ratio = ws_connected / n_samples
        er_connected_ratio = er_connected / n_samples
        
        if ws_connected_ratio < thr_connectivity or er_connected_ratio <thr_connectivity:
            continue # invalid configuration
        
        #Aggregate results
        mean_C_ws = np.mean(C_ws)
        mean_L_ws = np.mean(L_ws)
        mean_C_er = np.mean(C_er)
        mean_L_er = np.mean(L_er)
        
        # Small-worldness
        gamma_ = float(mean_C_ws / mean_C_er)
        lambda_ = float(mean_L_ws / mean_L_er)
        sigma_ = float(gamma_ / lambda_)
        if np.isnan(sigma_):
            continue
        
        lambdas.append(lambda_)
        gammas.append(gamma_)

        results.append((k, b, sigma_, gamma_, lambda_))
            
            
GAMMA_MIN = np.quantile(gammas, 0.5)
LAMBDA_MAX = np.quantile(lambdas, 0.5)
## We want to choose the configuration with the highest sigma (small-worldness)
filtered = [
    r for r in results
    if (r[3] >= GAMMA_MIN) and (r[4] <= LAMBDA_MAX)
]
filtered.sort(key=lambda x: x[2])
best = filtered[0]
print("Top configuration (k, beta, sigma):")
print(best) 

#### 2.2. Sample Graphs

In [ ]:
cfg_ws = {
    "seed": None,
    "network":
        {"generator": "WS",
         "params": {
        "n": N,
        "k": 8,
        "beta": 0.1
         }
        }
}

In [ ]:
X = []
check = []
for seed in seeds:
    cfg = cfg_ws.copy()
    cfg["seed"] = int(seed)
    G = Network(cfg, remap_seed=False)
    vec, n_components = graph2vec(G)
    X.append(vec)
    check.append(n_components)
X = np.array(X)
# Scale
Xz = StandardScaler().fit_transform(X)


selection_ids = maximin_select(Xz, K=8) 
pass_the_check = all(check[i] == 1 for i in selection_ids)
print(f"All selected graphs are connected: {pass_the_check}")

for i in selection_ids:
    r = X[i]
    print(f'{seeds[i]:>11} : mean-sp={r[0]:.3f}, glob-clustering-coeff={r[1]:.3f}, mean-deg={r[2]:>6.3f}, std-deg={r[3]:.3f}, nod0-deg={r[4]:.3f}')
    
seeds[selection_ids].tolist()

In [ ]:
# cfg_ws['seed'] = 1769674741
# G_ = Network(cfg_ws, remap_seed=False)
# import matplotlib.pyplot as plt
# plt.hist(np.sum(G_.adjacency_matrix(), axis=1), bins=5)

## (B) Sampling Statements

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler


def maximin_select_balanced(
    X: np.ndarray,
    K: int,
    labels: np.ndarray,
    init_selected: list[int] | None = None,
) -> list[int]:
    """Select K points with balanced binary labels via alternating maximin.

    Args:
        X: Feature matrix of shape (N, D).
        K: Even number of points to select.
        labels: Binary labels of shape (N,).
        init_selected: Optional pre-selected indices.

    Returns:
        Indices of selected points.

    Raises:
        ValueError: If K is odd, per-class quota exceeded, or not enough samples.
    """
    if K % 2:
        raise ValueError(f"K must be even, got {K}")
    quota = K // 2

    selected = list(init_selected or [])
    c0 = sum(labels[i] == 0 for i in selected)
    c1 = len(selected) - c0
    if c0 > quota or c1 > quota:
        raise ValueError("Pre-selected indices exceed per-class quota")

    if not selected:
        first = int(np.argmax(np.linalg.norm(X, axis=1)))
        selected.append(first)
        c0 += labels[first] == 0
        c1 += labels[first] == 1

    if np.sum(labels == 0) < quota or np.sum(labels == 1) < quota:
        raise ValueError("Insufficient samples to fill balanced quotas")

    while c0 < quota or c1 < quota:
        target = 0 if c0 < quota else 1
        dist = cdist(X, X[selected], metric="euclidean").min(axis=1)
        dist[selected] = -np.inf
        dist[labels != target] = -np.inf
        nxt = int(np.argmax(dist))
        selected.append(nxt)
        c0 += labels[nxt] == 0
        c1 += labels[nxt] == 1

    return selected

def maximin_select(
    X: np.ndarray, K: int, init_selected: list[int] | None = None
) -> list[int]:
    """Select K points from X using the maximin criterion.

    Args:
        X: Feature matrix of shape (N, D).
        K: Number of points to select.
        init_selected: Pre-selected point indices to include.

    Returns:
        List of selected point indices.
    """
    if init_selected is not None and len(init_selected) > 0:
        selected = list(init_selected)
        remaining = K - len(selected)
        if remaining <= 0:
            return selected[:K]
    else:
        selected = [int(np.argmax(np.linalg.norm(X, axis=1)))]
        remaining = K - 1

    for _ in range(remaining):
        distances = cdist(X, X[selected], metric="euclidean")
        min_dist = distances.min(axis=1)
        min_dist[selected] = -np.inf
        next_idx = np.argmax(min_dist)
        selected.append(int(next_idx))

    return selected

In [ ]:
from pathlib import Path

import polars as pl
import os
import sys
import numpy as np

sys.path.append(os.path.abspath('..'))

records_path = Path("../data/resources/predictions/")
models = ["llama-assistant", "llama-base", "llama-biomed", "llama-chemist", "llama-coder", "llama-cyber", "llama-finance", "llama-hermes", "llama-law", "llama-lexicographer",
          "llama-linguist", "llama-openmath", "llama-roleplay", "llama-scholar", "llama-user"]

In [ ]:
# Load and filter base data
df_doc = (
    pl.read_csv(records_path / "zs_med_test_split_llama-doc.csv")
    .filter(pl.col("negation") == 0, pl.col("real_object") == 1)
    .select(
        pl.col("").alias("init_idx"),
        "statement",
        pl.col("correct").alias("label_ground_truth"),
        pl.col("predicted_label").alias("doc_predicted_label"),
        pl.col("prob_true").alias("doc_prob"),
    )
    .with_row_index("idx")  # Adds 0-indexed sequential column

)

# Get valid indices for filtering
valid_idx = df_doc["init_idx"].to_list()

# Aggregate predictions across all models
dfs_models = [
    pl.read_csv(records_path / f"zs_med_test_split_{model}.csv")
    .filter(pl.col("").is_in(valid_idx)).rename({"": "init_idx"})
    .select("init_idx", "predicted_label", "prob_true", 'statement')
    .with_columns((pl.col('predicted_label')==1).cast(pl.Float32).alias('predicted_label'))
    for model in models
]

# Validate: ensure all model dataframes have same order as df_doc
for i, df_model in enumerate(dfs_models):
    model_indices = df_model["init_idx"].to_list()
    model_statements = df_model['statement'].to_list()
    assert model_indices == valid_idx, (
        f"Index mismatch in {models[i]}: model indices do not match df_doc order"
    )
    assert len(model_statements) == len(set(model_statements)), (
        f"Duplicate statements found in {models[i]}"
    )

# Concatenate and compute averages
df_agg = (
    pl.concat(dfs_models)
    .group_by("init_idx") # by init_idx
    .agg(
        pl.col("predicted_label").mean().alias("other_frac_predicted_label"),
        pl.col("prob_true").median().alias("other_median_prob"),
    ).with_row_index("idx")
)

# Join with base data
df_preds = df_doc.join(df_agg, on="idx", how="inner")

# Find initial statement indices
start_statements = [
    "Silver is indicated for the treatment of keratosis pilaris.",
    "Terbutaline is indicated for the treatment of cramps.",
    "Methyl nicotinate is indicated for the treatment of aches.",
    "Terbutaline is indicated for the treatment of asthma.",
]

#identify initial statements
start_idxs = df_preds.filter(pl.col("statement").is_in(start_statements))["idx"].to_list()

# Validation: ensure all rows matched
assert len(df_preds) == len(df_doc), (
    f"Join mismatch: expected {len(df_doc)} rows, got {len(df_preds)}"
)

# Validation: check that all start_statements are present
start_statements_found = (
    df_preds.filter(pl.col("statement").is_in(start_statements)).height
)
assert start_statements_found == len(start_statements), (
    f"Missing initial statements: found {start_statements_found}/{len(start_statements)}"
)

# additional metrics
df_preds = df_preds.with_columns(
    (1 - np.abs(pl.col("doc_prob") - pl.col("other_median_prob"))).cast(pl.Float64).alias("consensus_score"),
    (1 - (pl.col("label_ground_truth") - pl.col("other_frac_predicted_label")).abs()).cast(pl.Float64).alias("other_accuracy"),
    (1 - (pl.col('label_ground_truth') - pl.col('doc_prob')).abs()).cast(pl.Float64).alias("doc_accuracy"),
)
df_preds.head()

In [ ]:
LABELS_TO_USE = ['label_ground_truth', 'doc_predicted_label', 'doc_accuracy', 'other_accuracy', 'consensus_score']
X = df_preds.select(pl.col(LABELS_TO_USE)).to_numpy().astype(float)
scaler = StandardScaler()
Xz = scaler.fit_transform(X)
selected_ids = maximin_select_balanced(Xz, K=30, labels=X[:, 0], init_selected=start_idxs)
df_preds.filter(pl.col("idx").is_in(selected_ids)).select(['idx', 'statement'] + LABELS_TO_USE)

In [ ]:
# Create DataFrame with selection order
df_order = pl.DataFrame({
    "idx": selected_ids,
    "selection_order": range(len(selected_ids))
})

# Join and sort by selection order
df_selected = (
    df_preds
    .filter(pl.col("idx").is_in(selected_ids))
    .join(df_order, on="idx", how="inner")
    .sort("selection_order")
)
df_selected.select(["idx", "statement"] + LABELS_TO_USE)

#### Save to Configs

In [ ]:
raise RuntimeError("End of code snippet")
# just a saveguard again 'Run All'

In [ ]:
from pathlib import Path
from textwrap import dedent
import sys
import os
sys.path.append(os.path.abspath('..'))


import yaml

STAT_FIELDS = [
    "label_ground_truth",
    "doc_predicted_label",
    "doc_prob",
    "doc_accuracy",
    "other_frac_predicted_label",
    "other_accuracy",
    "other_median_prob",
    "consensus_score",
]

# Custom representer to force literal block scalar for multiline strings
def str_representer(dumper, data):
    if '\n' in data:
        return dumper.represent_scalar('tag:yaml.org,2002:str', data, style='|')
    return dumper.represent_scalar('tag:yaml.org,2002:str', data)

yaml.add_representer(str, str_representer)

def write_statement_configs(
    df: pl.DataFrame,
    selected_ids: list[int],
    out_dir: Path = Path("configs/statement"),
) -> list[Path]:
    """Write YAML configs for selected statements with sequential numbering per label.
    
    Each label group (true/false) gets independent sequential numbering starting from 0.
    Numbering respects selection_order within each label group.
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    desc = dedent("""\
    Auto-generated from maximin selection pipeline.
    
    Explaining fields in stats (doctor LLM is m42-health/Llama3-Med42-8B):
    - label_ground_truth: Statement label (binary, 1=True, 0=False)
    - doc_predicted_label: Predicted label by doctor LLM (binary, 1=True, 0=False)
    - doc_prob: Probability assigned by doctor LLM to statement being true
    - doc_accuracy: Accuracy of doctor LLM prediction vs ground truth
    - other_frac_predicted_label: Fraction of other LLMs predicting statement as true
    - other_accuracy: Accuracy of other LLM predictions vs ground truth
    - other_median_prob: Median probability from other LLMs that statement is true
    - consensus_score: Agreement between doctor LLM and other LLM probabilities (not accuracy!)
    
    To ensure diversity, statements were selected using a maximin criterion on the following features:
    - label_ground_truth
    - doc_predicted_label
    - doc_accuracy
    - other_accuracy
    - consensus_score
    
    """).strip()

    # Filter and sort by selection_order to preserve order
    df_filtered = df.filter(pl.col("idx").is_in(selected_ids)).sort("selection_order")
    
    # Track sequential IDs per label
    label_counters = {"false": 0, "true": 0}
    paths: list[Path] = []
    
    for row in df_filtered.to_dicts():
        label_key = "true" if row["label_ground_truth"] else "false"
        label_id = label_counters[label_key]
        label_counters[label_key] += 1
        
        cfg_id = f"{label_key}_{label_id}"
        cfg = {
            "defaults": ["base", "_self_"],
            "id": cfg_id,
            "statement": row["statement"],
            "label": {
                "correct": bool(row["label_ground_truth"]),
                "neither": False,
                "negated": False,
            },
            "description": desc,
            "stats": {
                k: (round(float(row[k]), 5) if isinstance(row[k], (int, float)) else row[k])
                for k in STAT_FIELDS
            },
        }
        path = out_dir / f"{cfg_id}.yaml"
        with path.open("w", encoding="utf-8") as f:
            yaml.dump(cfg, f, sort_keys=False, allow_unicode=True, default_flow_style=False)
        paths.append(path)

    return paths

# Write configs for selected statements, using df_selected order
out_paths = write_statement_configs(
    df=df_selected,
    selected_ids=selected_ids,
)
out_paths


## Diagnostics
1. Higher `min_pairwise_dist` → better spread.
2. Large per-feature variance/range → better coverage.
3. PCA ratios closer to uniform → more isotropic spread.

In [ ]:
import numpy as np
import polars as pl
from scipy.spatial.distance import pdist
from sklearn.decomposition import PCA


def diversity_report(df: pl.DataFrame, idx: list[int], feature_cols: list[str]) -> dict:
    """Compute simple diversity diagnostics on selected rows."""
    X = df.filter(pl.col("idx").is_in(idx)).select(feature_cols).to_numpy().astype(float)
    dists = pdist(X, metric="euclidean")
    pca = PCA().fit(X)
    return {
        "min_pairwise_dist": float(dists.min()),
        "mean_pairwise_dist": float(dists.mean()),
        "median_pairwise_dist": float(np.median(dists)),
        "feature_variance": dict(zip(feature_cols, X.var(axis=0).tolist())),
        "feature_range": dict(
            zip(feature_cols, (X.max(axis=0) - X.min(axis=0)).tolist())
        ),
        "pca_explained_var": pca.explained_variance_.tolist(),
        "pca_explained_ratio": pca.explained_variance_ratio_.tolist(),
    }

feature_cols = ['label_ground_truth', 'doc_predicted_label', 'doc_accuracy', 'other_accuracy', 'consensus_score']

report = diversity_report(df_preds, selected_ids, feature_cols)
report